In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
from app.core.parser import *
from app.core.loader import *
from app.reporting.history_report import *
import pandas as pd

In [3]:
pd.set_option('future.no_silent_downcasting', True)

#parent_dir = Path(__file__).resolve().parent
parent_dir = Path.cwd()
data = parent_dir / 'app' / 'data'

raw_dir = data / 'raw'
waterfall_dir =  raw_dir/ 'Waterfall'
consump_dir = raw_dir /  'Consumption'
proc_dir = data /  "Processed"
proc_cost = proc_dir / 'Cost.parquet'
cost_map = pd.read_parquet(proc_cost)

/var/folders/4x/k1x7fbgd5bl08h5y4f2kbghw0000gn/T/ipykernel_16448/4116792564.py:1: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


In [ ]:
proc_consum = proc_dir / 'Consumption'
consumption = all_consumption_load(proc_consum)

proc_wf = proc_dir / 'Waterfall'
minidx, maxidx = consumption.Orderidx.min(), consumption.Orderidx.max()
earliest_prediction = week_to_idx(2024, 1)
latest_prediction = week_to_idx(2026,10)
earliest_order = consumption.Orderidx.min()
latest_order = consumption.Orderidx.max()

wf = load_waterfall_span(earliest_prediction, 
                         latest_prediction,
                         proc_wf)
#wf = wf[[col for col in wf.columns if not str(col).replace('.', '', 1).isdigit() ]]
# wf = wf[(wf.Orderidx<=latest_order)&(wf.Orderidx>=earliest_order)]

# consumption = consumption[consumption.Orderidx<=latest_order]
wf, consumption = filter_disjoint(wf, consumption)
joint = wf.merge(consumption, on=["Part", 'Orderidx'], how="outer")

# Filter out instances where PredQty is never non-NaN; ie consumed parts that are not waterfall parts. 
filtered = joint.groupby('Part').filter(lambda g: g['PredQty'].notna().any())
filtered.Quantity1 = filtered.Quantity1.fillna(0)
filtered.rename({"Quantity1":"OrderQty"}, axis=1, inplace=True)
consumption.rename({"Quantity1":"OrderQty"}, axis=1, inplace=True)

filtered.PredQty = filtered.PredQty.fillna(0)

#report.to_csv(parent_dir / 'HistoryReport.csv')
# errorReport(filtered, save_loc=Path(Path.cwd() /'reports'))

In [40]:
consumption.sort_values("Orderidx")

,Part,Orderidx,OrderQty
82460,86511580,105288,0.0
82341,85665922,105288,20.0
82340,85665921,105288,1.0
82339,85665920,105288,0.0
82338,85665919,105288,42.0
...,...,...,...
96212,86307540,105361,0.0
96211,86307536,105361,0.0
96210,86307530,105361,0.0
96217,86307557,105361,0.0


In [43]:
wf.sort_values("RelDate")


,Part,Orderidx,Predidx,RelDate,01002316,01007780,01009399,01009600,01011153,01011944,...,01012448,01012451,01012449,01012445,01014101,01014108,01014109,01014110,01014147,01012425
43448,85598071,105262,105249,2024-01-02,0.0,0.0,1.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16959,84824527,105271,105249,2024-01-02,0.0,0.0,1.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22966,84857951,105271,105249,2024-01-02,0.0,0.0,1.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20561,84824559,105267,105249,2024-01-02,0.0,0.0,1.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14724,84824518,105261,105249,2024-01-02,0.0,0.0,1.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8223332,M0694835-103H,105382,105368,2026-04-14,NaN,0.0,0.0,NaN,NaN,0.0,...,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,0.0,0.0
7966980,M0694835-103H,105379,105368,2026-04-14,NaN,0.0,0.0,NaN,NaN,0.0,...,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,0.0,0.0
7967062,M0760544-103H,105379,105368,2026-04-14,NaN,0.0,0.0,NaN,NaN,0.0,...,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,0.0,0.0
8058799,M0694835-103H,105380,105368,2026-04-14,NaN,0.0,0.0,NaN,NaN,0.0,...,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,0.0,0.0


In [ ]:
shipcols = [col for col in wf.columns if str(col).replace('.', '', 1).isdigit() ]

# filter columns to those with sum >= 5


wf.iloc[wf['01009600'].argmax()]

Part                   84699760
Orderidx                 105289
Predidx                  105289
RelDate     2024-10-07 00:00:00
01002316                    0.0
01007780                    0.0
01009399                    0.0
01009421                    0.0
01009600                    1.0
01011153                    NaN
01011944                    0.0
01012188                    0.0
01012234                    0.0
01012318                    0.0
01012369                    0.0
01012371                    0.0
01012372                    0.0
01012426                    0.0
01012445                    0.0
01012446                    0.0
01012447                    0.0
01012448                    0.0
01012449                    0.0
01012451                    0.0
01012452                    0.0
01012454                    0.0
01012455                    0.0
01014002                    0.0
10000002                    0.0
PredQty                     282
PCR                    144024.0
01014101

In [7]:
filtered

,Part,Orderidx,Predidx,RelDate,PredQty,PCR,OrderQty
0,24002551,105289,105290.0,2024-10-14,60.0,62.0,0.0
1,24002551,105290,105291.0,2024-10-21,60.0,62.0,0.0
2,24002551,105291,105281.0,2024-08-12,1.0,42.0,0.0
3,24002551,105291,105282.0,2024-08-19,1.0,42.0,0.0
4,24002551,105291,105283.0,2024-08-26,1.0,42.0,0.0
...,...,...,...,...,...,...,...
470579,M0760544-103H,105361,105348.0,2025-11-26,72.0,26832.0,0.0
470580,M0760544-103H,105361,105349.0,2025-12-03,84.0,26832.0,0.0
470581,M0760544-103H,105361,105350.0,2025-12-10,72.0,26832.0,0.0
470582,M0760544-103H,105361,105351.0,2025-12-16,72.0,26832.0,0.0


In [16]:
week_to_idx(2025,1)

105301

In [15]:
from app.models.train import _expanding_stats
from app.models.train import *

_expanding_stats(filtered)

,_mean,_std,_min,_max,_count,_cv
2,NaN,NaN,NaN,NaN,0.0,NaN
3,1.000000,NaN,1.0,1.0,1.0,NaN
4,1.000000,0.000000,1.0,1.0,2.0,0.000000
5,1.000000,0.000000,1.0,1.0,3.0,0.000000
6,1.000000,0.000000,1.0,1.0,4.0,0.000000
...,...,...,...,...,...,...
470567,325.942413,413.139398,1.0,6240.0,1094.0,1.267523
470582,325.721461,413.015256,1.0,6240.0,1095.0,1.268001
470568,325.489964,412.897754,1.0,6240.0,1096.0,1.268542
470583,325.204193,412.817865,1.0,6240.0,1097.0,1.269411


In [8]:
from app.models.train import *

features = time_series_features(filtered,consumption)

In [9]:
features

,Part,Orderidx,Predidx,PredQty,PCR,OrderQty,Lookahead,Pred_CumCount,RelQty_lag_1wk,RelQty_lag_2wk,...,pn3,pn4,pn5,pn6,pn7,L,M,ReleaseMonth,ReleaseDay,ReleaseWD
0,24002551,105289.0,105290.0,60.0,62.0,0.0,-1.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,288.0,0.0
1,24002551,105290.0,105291.0,60.0,62.0,0.0,-1.0,0.0,60.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,295.0,0.0
2,24002551,105291.0,105281.0,1.0,42.0,0.0,10.0,0.0,60.0,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,225.0,0.0
3,24002551,105291.0,105282.0,1.0,42.0,0.0,9.0,1.0,1.0,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,232.0,0.0
4,24002551,105291.0,105283.0,1.0,42.0,0.0,8.0,2.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,239.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470579,M0760544-103H,105361.0,105348.0,72.0,26832.0,0.0,13.0,10.0,108.0,96.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,11.0,330.0,2.0
470580,M0760544-103H,105361.0,105349.0,84.0,26832.0,0.0,12.0,11.0,72.0,108.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,12.0,337.0,2.0
470581,M0760544-103H,105361.0,105350.0,72.0,26832.0,0.0,11.0,12.0,84.0,72.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,12.0,344.0,2.0
470582,M0760544-103H,105361.0,105351.0,72.0,26832.0,0.0,10.0,13.0,72.0,84.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,12.0,350.0,1.0


In [51]:
df = features.copy()

normal_pn = df.Part.str.len()==8
part_no_cols = [f'pn{i}' for  i in range(8)]
df.Part[normal_pn].str.split('',expand=True)


,0,1,2,3,4,5,6,7,8,9
0,,2,4,0,0,2,5,5,1,
1,,2,4,0,0,2,5,5,1,
2,,2,4,0,0,2,5,5,1,
3,,2,4,0,0,2,5,5,1,
4,,2,4,0,0,2,5,5,1,
...,...,...,...,...,...,...,...,...,...,...
581419,,8,7,8,6,5,9,6,0,
581420,,8,7,8,6,5,9,6,0,
581421,,8,7,8,6,5,9,6,0,
581422,,8,7,8,6,5,9,6,0,


count                                                   ...  \
bins            -1.0 -0.9 -0.8  -0.7  -0.6 -0.5  -0.4 -0.3 -0.2  -0.1  ...   
Part                                                                   ...   
24002551       167.0  2.0  0.0   0.0   0.0  0.0   0.0  0.0  6.0   0.0  ...   
24002555       291.0  0.0  0.0   0.0   0.0  0.0   0.0  0.0  0.0   0.0  ...   
24002618        12.0  0.0  0.0   0.0   0.0  0.0   0.0  0.0  0.0   0.0  ...   
24002621       148.0  0.0  0.0   0.0   0.0  1.0   0.0  0.0  1.0   0.0  ...   
24002989         1.0  0.0  0.0   0.0   0.0  0.0   0.0  0.0  0.0   0.0  ...   
...              ...  ...  ...   ...   ...  ...   ...  ...  ...   ...  ...   
L0319584AA01     0.0  0.0  0.0   0.0   0.0  0.0   0.0  0.0  0.0   0.0  ...   
L0333882AB01    11.0  0.0  0.0   0.0   0.0  0.0   0.0  0.0  0.0   0.0  ...   
L0333884AB01    36.0  0.0  0.0   0.0   0.0  0.0   0.0  0.0  0.0   0.0  ...   
M0694835-103H  652.0  7.0  1.0   4.0  16.0  9.0  15.0  3.0  4.0  10.0  ...   
M0760544-103H  912.0  0.0  0.0  15.0   6.0  5.0   3.0  1.0  2.0   1.0  ...   

                                                                          
bins           0.1   0.2   0.3   0.4   0.5   0.6   0.7   0.8   0.9   1.0  
Part                                                                      
24002551       1.0   6.0   0.0   0.0   0.0   0.0   2.0   6.0   0.0   0.0  
24002555       0.0   1.0   0.0   0.0   0.0   0.0   4.0   0.0   0.0   0.0  
24002618       0.0   0.0   0.0   0.0   0.0   2.0   0.0   0.0   0.0   0.0  
24002621       1.0   1.0   0.0   0.0   1.0   1.0   0.0   0.0   2.0   0.0  
24002989       0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
...            ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  
L0319584AA01   0.0   0.0   0.0   0.0   0.0   0.0   2.0   0.0   0.0   0.0  
L0333882AB01   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
L0333884AB01   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
M0694835-103H  5.0  13.0  20.0  26.0  29.0  44.0  19.0  34.0  10.0   1.0  
M0760544-103H  3.0   7.0   4.0   7.0  29.0  53.0  60.0  58.0  20.0  11.0  

[1786 rows x 21 columns]

In [95]:
latest

,Part,Orderidx,Predidx,RelDate,PredQty,PCR,Quantity1
2,24002551,105291,105281.0,2024-08-12,1.0,42.0,0.0
3,24002551,105291,105282.0,2024-08-19,1.0,42.0,0.0
4,24002551,105291,105283.0,2024-08-26,1.0,42.0,0.0
5,24002551,105291,105285.0,2024-09-09,1.0,42.0,0.0
6,24002551,105291,105286.0,2024-09-16,1.0,42.0,0.0
...,...,...,...,...,...,...,...
656633,M0760544-103H,105366,105367.0,2026-04-07,60.0,569.0,0.0
656654,M0760544-103H,105367,105367.0,2026-04-07,744.0,569.0,0.0
656675,M0760544-103H,105368,105367.0,2026-04-07,300.0,569.0,0.0
656655,M0760544-103H,105367,105368.0,2026-04-14,264.0,569.0,0.0


In [57]:
wf.Orderidx.min(), consumption.Orderidx.min()

(np.int64(105289), np.int64(105288))

In [ ]:

merged = cf.merge(part_pivot, on=["Part", oi], how="inner")
#merged.rename(columns={'Qty': pq}, inplace=True)

print("Initial count of instances: ", merged.shape[0])
filtered = filterZeroDemand(merged, 
                        q1=shipping_cols, 
                        q2=oq)
print('After demand filter: ', filtered.shape[0])

# Infill zero for non-prediction/non-consumption 
filtered.loc[list(filtered[filtered.Predidx.isna()].index), pq] = 0
filtered.loc[list(filtered[filtered.Orderidx.isna()].index), oq] = 0
filtered = filtered.dropna(subset=[pq, oq], how='all', axis=0)


Replaced data in  2024_Wk_18.parquet
Replaced data in  2024_Wk_19.parquet
Replaced data in  2024_Wk_20.parquet
Replaced data in  2024_Wk_21.parquet
Replaced data in  2024_Wk_22.parquet
Replaced data in  2024_Wk_23.parquet
Replaced data in  2024_Wk_24.parquet
Replaced data in  2024_Wk_25.parquet
Replaced data in  2024_Wk_26.parquet
Replaced data in  2024_Wk_27.parquet
Replaced data in  2024_Wk_28.parquet
Replaced data in  2024_Wk_29.parquet
Replaced data in  2024_Wk_30.parquet
Replaced data in  2024_Wk_31.parquet
Replaced data in  2024_Wk_32.parquet
Replaced data in  2024_Wk_33.parquet
Replaced data in  2024_Wk_34.parquet
Replaced data in  2024_Wk_35.parquet
Replaced data in  2024_Wk_36.parquet
Replaced data in  2024_Wk_37.parquet
Replaced data in  2024_Wk_38.parquet
Replaced data in  2024_Wk_39.parquet
Replaced data in  2024_Wk_40.parquet


In [57]:
weekdata

,Part,Orderidx,Predidx,RelDate,01002316,01007780,01009399,01009421,01009600,01011153,...,01012448,01012449,01012451,01012452,01012454,01012455,01014002,10000002,PredQty,PCR
31,24002551,105289,105290,2024-10-14,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60,62.0
43,24002555,105289,105285,2024-09-09,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,28.0
44,24002555,105289,105286,2024-09-16,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,28.0
45,24002555,105289,105287,2024-09-23,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,28.0
46,24002555,105289,105288,2024-09-30,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,48.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86955,M0760544-103H,105289,105276,2024-07-10,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,480,4260.0
86956,M0760544-103H,105289,105277,2024-07-17,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,384,4260.0
86957,M0760544-103H,105289,105278,2024-07-24,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,384,4740.0
86958,M0760544-103H,105289,105279,2024-07-31,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,432,7344.0


In [ ]:

# merged = cf.merge(part_pivot, on=["Part", oi], how="inner")
#merged.rename(columns={'Qty': pq}, inplace=True)

# print("Initial count of instances: ", merged.shape[0])
# filtered = filterZeroDemand(merged, 
#                         q1=shipping_cols, 
#                         q2=oq)
# print('After demand filter: ', filtered.shape[0])

# # if no prediction was made, N weeks prior were 0. 
# filtered.loc[list(filtered[filtered.Predidx.isna()].index), pq] = 0
# # if no prediction was made, N weeks prior were 0. 
# filtered.loc[list(filtered[filtered.Orderidx.isna()].index), oq] = 0
# filtered = filtered.dropna(subset=[pq, oq], how='all', axis=0)

# # # do we want to add more cases for thsis ? augment more predictions to show gaps
# # filtered.loc[filtered.Predidx.isna(), "PredQty"] = 0  
# # filtered.loc[filtered.Predidx.isna(), "Predidx"] = filtered.loc[filtered.Predidx.isna(), "Orderidx"].values

# filtered.to_csv(outpath, index=False, sep='|')


/Users/nikkivanhandel/Research/SupplyChainForecasting/data/raw/Consumption/2024/24wk40.prn


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa0 in position 151856: invalid start byte